### Ячейка 1: Импорты

In [11]:
import os
import random
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import OneCycleLR, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import Levenshtein
import hydra
from omegaconf import DictConfig, OmegaConf, MISSING # MISSING для обязательных полей

# --- HPO ---
import optuna

# --- Визуализация и утилиты ---
from tqdm.notebook import tqdm # Используем tqdm для ноутбуков
import matplotlib.pyplot as plt

# --- Настройка логирования ---
# Hydra обычно настраивает логирование, но можно задать базовую конфигурацию
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
log = logging.getLogger(__name__)

# --- Подавление предупреждений (опционально) ---
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Проверка доступности GPU ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Using device: {DEVICE}")

Using device: cuda


### Ячейка 2: Конфигурация с Hydra (Dataclasses)

In [12]:
# --- Ячейка 2: Конфигурация с Hydra (Dataclasses) - ОБНОВЛЕНО (Optional типы) ---

import os
import random
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass, field
# !!! Импортируем Optional !!!
from typing import List, Tuple, Dict, Any, Optional
import pprint

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import OneCycleLR, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import Levenshtein
import hydra
from omegaconf import DictConfig, OmegaConf, MISSING

import optuna
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# --- Логирование и настройки без изменений ---
# ... (код логгера) ...
log_formatter = logging.Formatter('%(message)s')
log_handler = logging.StreamHandler()
log_handler.setFormatter(log_formatter)
log = logging.getLogger(__name__)
log.setLevel(logging.INFO)
if log.hasHandlers():
    log.handlers.clear()
log.addHandler(log_handler)
log.propagate = False
logging.getLogger("optuna").setLevel(logging.WARNING)
logging.getLogger("hydra").setLevel(logging.WARNING)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- Конец настроек ---


# --- Конфигурационные классы ---
@dataclass
class PathsConfig:
    data_dir: str = MISSING
    feature_dir: str = MISSING

@dataclass
class DataParamsConfig:
    label_column: str = "message"
    metadata_filename: str = "train.csv"
    feature_input_channels: int = 18
    feature_seq_len: int = 4000
    train_val_split_ratio: float = 0.1
    random_seed: int = 42
    blank_char: str = "<blank>"
    allowed_chars: str = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,?"
    dataset_fraction: float = 1.0

@dataclass
class ModelParamsConfig:
    block_out_channels: List[int] = field(default_factory=lambda: [64, 128])
    block_kernel_size_0: int = 5
    block_kernel_size_1: int = 3
    block_stride_0: int = 2
    block_stride_1: int = 2
    activation_type: str = 'GELU'
    process_channels_separately: bool = True
    rnn_type: str = 'GRU'
    rnn_hidden_size: int = 256
    rnn_num_layers: int = 2
    rnn_dropout: float = 0.2
    classifier_dropout: float = 0.2
    vocab_size: Optional[int] = None # Уже был Optional, это правильно

@dataclass
class TrainParamsConfig:
    batch_size: int = 32
    num_epochs: int = 50
    learning_rate: float = 1e-4
    max_lr: float = 3e-4
    optimizer: str = 'AdamW'
    scheduler: str = 'OneCycleLR'
    scheduler_reduce_factor: float = 0.1
    scheduler_patience: int = 5
    weight_decay: float = 1e-4
    use_amp: bool = True
    early_stopping_patience: int = 10
    max_grad_norm: Optional[float] = 1.0 # Уже был Optional, это правильно

    # --- ИЗМЕНЕНЫ ТИПЫ НА Optional[int] ---
    # Позволяет использовать null в YAML для авто-расчета.
    # Устанавливаем default=None, что логично для авто-расчета.
    levenshtein_slope_window: Optional[int] = None
    min_epochs_for_slope: Optional[int] = None
    levenshtein_slope_percentage: float = 0.4 # Тип float, остается без изменений

@dataclass
class HPOParamsConfig:
    n_trials: int = 50
    storage_name: Optional[str] = None # Уже был Optional
    study_name: Optional[str] = None # Уже был Optional
    metric_to_optimize: str = "val_levenshtein"
    direction: str = "minimize"
    param_distributions: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Config:
    defaults: List[Any] = field(default_factory=lambda: [
        "_self_",
        {"override hydra/hydra_logging": "disabled"},
        {"override hydra/job_logging": "disabled"},
    ])
    paths: PathsConfig = field(default_factory=PathsConfig)
    data_params: DataParamsConfig = field(default_factory=DataParamsConfig)
    model_params: ModelParamsConfig = field(default_factory=ModelParamsConfig)
    train_params: TrainParamsConfig = field(default_factory=TrainParamsConfig) # Теперь с Optional типами
    hpo_params: HPOParamsConfig = field(default_factory=HPOParamsConfig)
    enable_hpo: bool = True
    finetune_checkpoint_path: Optional[str] = None # Уже был Optional
    run_timestamp: Optional[str] = None # Уже был Optional

cs = hydra.core.config_store.ConfigStore.instance()
cs.store(name="base_config", node=Config)

log.info("Конфигурационные классы обновлены (используется Optional[int] для параметров наклона).")
log.info(f"Используемое устройство: {DEVICE}")

Конфигурационные классы обновлены (используется Optional[int] для параметров наклона).
Используемое устройство: cuda


### Ячейка 3: Функции Загрузки и Предобработки Данных

In [13]:
### Ячейка 3: Функции Загрузки и Предобработки Данных (Русские ошибки/предупреждения)

def set_seed(seed_value: int):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

def load_metadata(data_dir: Path, cfg: DictConfig) -> pd.DataFrame:
    filename = cfg.data_params.metadata_filename
    label_col = cfg.data_params.label_column
    filepath = data_dir / filename
    if not filepath.exists():
        log.error(f"🛑 Ошибка: Файл метаданных не найден: {filepath}")
        raise FileNotFoundError(f"Файл метаданных не найден: {filepath}")
    df = pd.read_csv(filepath)
    if label_col not in df.columns:
        log.error(f"🛑 Ошибка: Колонка с метками '{label_col}' не найдена в {filepath}. Доступные колонки: {df.columns.tolist()}")
        raise KeyError(f"Колонка с метками '{label_col}' не найдена.")
    df[label_col] = df[label_col].astype(str).str.upper().str.strip()
    df = df.dropna(subset=[label_col])
    return df

def create_vocabulary(df: pd.DataFrame, cfg: DictConfig) -> Tuple[Dict[str, int], Dict[int, str], int]:
    labels = df[cfg.data_params.label_column]
    allowed_chars = cfg.data_params.allowed_chars
    blank_char = cfg.data_params.blank_char
    unique_chars = set(allowed_chars)
    all_data_chars = set("".join(labels.tolist()))
    unknown_chars = all_data_chars - unique_chars
    if unknown_chars:
        log.warning(f"⚠️ Предупреждение: Найдены неизвестные символы в метках (не добавлены в allowed_chars): {unknown_chars}. Они будут проигнорированы.")
    char_list = sorted(list(unique_chars))
    char_list.append(blank_char)
    char2idx = {char: idx for idx, char in enumerate(char_list)}
    idx2char = {idx: char for idx, char in enumerate(char_list)}
    vocab_size = len(char_list)
    return char2idx, idx2char, vocab_size

def split_data(df: pd.DataFrame, test_size: float, random_state: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if test_size == 0:
        return df, pd.DataFrame(columns=df.columns)
    elif 0 < test_size < 1:
        train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True)
        return train_df, val_df
    else:
        log.error("🛑 Ошибка: train_val_split_ratio должен быть между 0 и 1.")
        raise ValueError("train_val_split_ratio должен быть между 0 и 1.")

### Ячейка 4: Класс Dataset

In [14]:
### Ячейка 4: Класс Dataset (Русские ошибки/предупреждения)

class MorseDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_dir: Path, char2idx: Dict[str, int], cfg: DictConfig):
        self.df_orig = df
        self.feature_dir = feature_dir
        self.char2idx = char2idx
        self.seq_len = cfg.data_params.feature_seq_len
        self.allowed_chars_set = set(cfg.data_params.allowed_chars)
        self.label_column = cfg.data_params.label_column
        self.blank_char_idx = char2idx.get(cfg.data_params.blank_char, -1)

        if self.label_column not in df.columns:
             log.error(f"🛑 Ошибка Dataset: Колонка '{self.label_column}' не найдена в DataFrame. Доступные: {df.columns.tolist()}")
             self.df = pd.DataFrame(columns=df.columns)
             return

        original_len = len(self.df_orig)
        self.df = self.df_orig[self.df_orig[self.label_column].apply(lambda text: all(c in self.char2idx for c in str(text)))]
        filtered_len = len(self.df)

        if original_len > filtered_len:
             log.warning(f"⚠️ Dataset: Отфильтровано {original_len - filtered_len} сэмплов из-за неизвестных символов в колонке '{self.label_column}'.")
        if filtered_len == 0 and original_len > 0:
             log.error(f"🛑 Dataset: Все сэмплы были отфильтрованы! Проверьте allowed_chars, label_column ('{self.label_column}') и данные.")

        self.df = self.df.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if idx >= len(self.df):
             log.error(f"🛑 Dataset: Индекс {idx} вне диапазона (длина: {len(self.df)})")
             return torch.empty(0), torch.empty(0)
        try:
            sample = self.df.iloc[idx]
            filename_raw = sample.get('id', None)
            if filename_raw is None:
                log.error(f"🛑 Dataset: Колонка 'id' не найдена для индекса {idx}. Ключи: {sample.keys()}")
                return torch.empty(0), torch.empty(0)
            label_text = str(sample[self.label_column])
            base_filename = str(filename_raw).split('.')[0]
            feature_path = self.feature_dir / f"{base_filename}.npy"
            features = np.load(feature_path).astype(np.float32)
            if features.shape[1] != self.seq_len:
                 if features.shape[1] < self.seq_len:
                     padding = np.zeros((features.shape[0], self.seq_len - features.shape[1]), dtype=np.float32)
                     features = np.concatenate((features, padding), axis=1)
                 else:
                     features = features[:, :self.seq_len]
            features_tensor = torch.from_numpy(features)
            label_indices = [self.char2idx[char] for char in label_text if char in self.char2idx]
            label_tensor = torch.tensor(label_indices, dtype=torch.long)
            return features_tensor, label_tensor
        except FileNotFoundError:
            log.error(f"🛑 Dataset: Файл признаков не найден: {feature_path}. Пропуск сэмпла {idx}.")
            return torch.empty(0), torch.empty(0)
        except Exception as e:
            log.error(f"🛑 Dataset: Ошибка обработки сэмпла {idx} (файл: {base_filename}): {e}")
            return torch.empty(0), torch.empty(0)

### Ячейка 5: Функция collate_fn

In [15]:
### Ячейка 5: Функция collate_fn (Без изменений)
# Эта ячейка не имела логов для перевода
def collate_fn(batch: List[Tuple[torch.Tensor, torch.Tensor]], blank_idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    batch = [item for item in batch if item[0].numel() > 0 and item[1].numel() > 0]
    if not batch:
        return torch.empty(0), torch.empty(0), torch.empty(0), torch.empty(0)
    features = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    features_stacked = torch.stack(features, dim=0)
    feature_lengths = torch.full((len(batch),), features_stacked.shape[2], dtype=torch.long)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=blank_idx)
    label_lengths = torch.tensor([len(lab) for lab in labels], dtype=torch.long)
    return features_stacked, labels_padded, feature_lengths, label_lengths

### Ячейка 6: Архитектура Модели (CRNN)

In [16]:
# Ячейка 6: Архитектура Модели (CRNN с ResNet - Индивидуальные параметры слоев) - ОБНОВЛЕНО

# --- Вспомогательная функция для получения активации (без изменений) ---
def get_activation(activation_type: str):
    # ... (код get_activation) ...
    if activation_type.lower() == 'relu':
        return nn.ReLU(inplace=True)
    elif activation_type.lower() == 'gelu':
        return nn.GELU()
    else:
        log.warning(f"Неизвестный тип активации '{activation_type}'. Используется GELU.")
        return nn.GELU()

# --- Обновленный Класс Остаточного Блока (без изменений) ---
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, groups: int = 1, activation_type: str = 'GELU'):
        super().__init__()
        # ... (код ResidualBlock1D) ...
        self.stride = stride
        self.in_channels = in_channels
        self.out_channels = out_channels
        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding, groups=groups, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.activation = get_activation(activation_type) # Используем GELU или ReLU
        # Вторая свертка всегда groups=1
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=1, padding=padding, groups=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        # Shortcut всегда groups=1
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, groups=1, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ... (код forward) ...
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.activation(out) # Применяем активацию
        out = self.conv2(out)
        out = self.bn2(out)
        identity_transformed = self.shortcut(identity)
        out += identity_transformed
        out = self.activation(out) # Применяем активацию после сложения
        return out


# --- Обновленный MorseRecognizer1D (Использует индивидуальные параметры слоев) ---
class MorseRecognizer1D(nn.Module):
    def __init__(self, cfg: DictConfig):
        super().__init__()
        self.model_cfg = cfg.model_params
        self.data_cfg = cfg.data_params
        self.vocab_size = self.model_cfg.vocab_size

        if self.vocab_size is None:
             raise ValueError("🛑 Ошибка: vocab_size должен быть установлен в конфиге перед инициализацией модели.")

        in_channels = self.data_cfg.feature_input_channels
        current_seq_len = float(self.data_cfg.feature_seq_len)

        # --- Residual Blocks ---
        res_blocks = []
        current_channels = in_channels
        num_stages = len(self.model_cfg.block_out_channels) # Определяем кол-во стадий по каналам
        activation_type = self.model_cfg.activation_type

        # ИЗМЕНЕНО: Убрана проверка длин списков kernel/stride

        log.info("--- Создание ResNet блоков ---")
        for i in range(num_stages):
            block_out_c = self.model_cfg.block_out_channels[i]

            # ИЗМЕНЕНО: Получаем kernel и stride для ТЕКУЩЕГО слоя 'i'
            try:
                block_k = getattr(self.model_cfg, f'block_kernel_size_{i}')
                block_s = getattr(self.model_cfg, f'block_stride_{i}')
            except AttributeError as e:
                 log.error(f"🛑 Ошибка Конфига: Не найден параметр '{e.name}' для слоя {i} в model_params.")
                 raise AttributeError(f"Не найден параметр '{e.name}' для слоя {i} в model_params.") from e


            block_groups = 1
            if i == 0 and self.model_cfg.process_channels_separately:
                if current_channels % 2 == 0 and block_out_c % 2 == 0:
                    block_groups = 2
                    log.info(f"  > Блок {i}: groups=2 (In={current_channels}, Out={block_out_c}, K={block_k}, S={block_s})")
                else:
                    log.warning(f"⚠️ Блок {i}: Невозможно groups=2 (In={current_channels}, Out={block_out_c}). Используется groups=1.")
                    log.info(f"  > Блок {i}: groups=1 (In={current_channels}, Out={block_out_c}, K={block_k}, S={block_s})")
            else:
                 log.info(f"  > Блок {i}: groups=1 (In={current_channels}, Out={block_out_c}, K={block_k}, S={block_s})")


            res_block = ResidualBlock1D(
                in_channels=current_channels,
                out_channels=block_out_c,
                kernel_size=block_k, # Используем полученное значение
                stride=block_s,      # Используем полученное значение
                groups=block_groups,
                activation_type=activation_type
            )
            res_blocks.append(res_block)
            current_channels = block_out_c

            if block_s > 1:
                 current_seq_len = np.ceil(current_seq_len / block_s)
                 log.info(f"    -> Длина посл. после блока {i}: {int(current_seq_len)}")

        self.res_blocks = nn.Sequential(*res_blocks)
        log.info("--- ResNet блоки созданы ---")

        # --- Расчет фактора сжатия (без изменений) ---
        final_cnn_output_len = int(current_seq_len)
        self._time_reduction_factor = self.data_cfg.feature_seq_len / final_cnn_output_len if final_cnn_output_len > 0 else float('inf')

        # --- RNN Слой (без изменений) ---
        rnn_input_size = current_channels
        rnn_class = nn.GRU if self.model_cfg.rnn_type == 'GRU' else nn.LSTM
        self.rnn = rnn_class(
            input_size=rnn_input_size,
            hidden_size=self.model_cfg.rnn_hidden_size,
            num_layers=self.model_cfg.rnn_num_layers,
            dropout=self.model_cfg.rnn_dropout if self.model_cfg.rnn_num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True
        )

        # --- Классификатор (без изменений) ---
        self.classifier = nn.Sequential(
            nn.Dropout(self.model_cfg.classifier_dropout),
            nn.Linear(self.model_cfg.rnn_hidden_size * 2, self.vocab_size)
        )

    def get_time_reduction_factor(self) -> float:
        # ... (код get_time_reduction_factor без изменений) ...
        if self._time_reduction_factor <= 0 or self._time_reduction_factor == float('inf'):
             log.warning(f"⚠️ Некорректный фактор сжатия времени ({self._time_reduction_factor}). Возвращаю 1.0. Проверьте страйды CNN и длину входа.")
             return 1.0
        return self._time_reduction_factor

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ... (код forward без изменений) ...
        x = self.res_blocks(x) # (B, C_out, L_out)
        x = x.permute(0, 2, 1) # (B, L_out, C_out)
        x, _ = self.rnn(x)     # (B, L_out, H*2)
        x = self.classifier(x) # (B, L_out, V)
        x = x.permute(1, 0, 2) # (L_out, B, V) - формат для CTCLoss
        return x

log.info("Класс модели MorseRecognizer1D обновлен для использования индивидуальных параметров слоев.")

Класс модели MorseRecognizer1D обновлен для использования индивидуальных параметров слоев.


### Ячейка 7: Утилиты (Декодер, Метрика, Анализ ошибок)

In [17]:
# --- Ячейка 7: Утилиты (Русские ошибки/предупреждения + Наклон Levenshtein) ---

import Levenshtein
import numpy as np
import torch
from typing import List, Dict, Tuple, Optional
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from scipy import stats # <-- Импорт Scipy для линейной регрессии

# (Функции ctc_greedy_decode, calculate_levenshtein, decode_targets без изменений)
def ctc_greedy_decode(logits: torch.Tensor, idx2char: Dict[int, str], blank_idx: int) -> List[str]:
    # ... (код без изменений) ...
    decoded_batch = []
    best_path = torch.argmax(logits, dim=2)
    best_path = best_path.transpose(0, 1).cpu().numpy()
    for sequence in best_path:
        collapsed = [seq for i, seq in enumerate(sequence) if i == 0 or seq != sequence[i-1]]
        decoded_sequence = [idx2char[idx] for idx in collapsed if idx != blank_idx]
        decoded_batch.append("".join(decoded_sequence))
    return decoded_batch

def calculate_levenshtein(predictions: List[str], targets: List[str]) -> float:
    # ... (код без изменений) ...
    total_distance = 0
    if not predictions or not targets: return 0.0 # Добавим проверку на пустые таргеты
    # Убедимся что длины совпадают для zip
    min_len = min(len(predictions), len(targets))
    if len(predictions) != len(targets):
        log.warning(f"⚠️ calculate_levenshtein: Несовпадение длин списков предсказаний ({len(predictions)}) и целей ({len(targets)}). Считаем по {min_len} парам.")

    for i in range(min_len):
        pred = predictions[i]
        target = targets[i]
        total_distance += Levenshtein.distance(pred, target)
    return total_distance / min_len if min_len > 0 else 0.0


def decode_targets(target_indices: torch.Tensor, target_lengths: torch.Tensor, idx2char: Dict[int, str]) -> List[str]:
    # ... (код без изменений) ...
    decoded_targets = []
    target_indices_cpu = target_indices.cpu().numpy()
    target_lengths_cpu = target_lengths.cpu().numpy()
    for i in range(target_indices_cpu.shape[0]):
        true_len = target_lengths_cpu[i]
        indices = target_indices_cpu[i, :true_len]
        text = "".join([idx2char.get(idx, '?') for idx in indices]) # Используем get для безопасности
        decoded_targets.append(text)
    return decoded_targets

# --- НОВАЯ ФУНКЦИЯ ---
def calculate_levenshtein_slope(history: List[float], k: int) -> Optional[float]:
    """
    Рассчитывает наклон кривой Levenshtein за последние K точек
    с использованием линейной регрессии (scipy.stats.linregress).

    Args:
        history: Список значений val_levenshtein по эпохам.
        k: Размер окна (количество последних эпох).

    Returns:
        Значение наклона или None, если данных недостаточно или произошла ошибка.
    """
    if len(history) < max(k, 2): # Нужно хотя бы 2 точки для линии, и хотя бы k точек
        # log.debug(f"Недостаточно данных для расчета наклона (нужно {max(k, 2)}, есть {len(history)})")
        return None
    y = np.array(history[-k:])
    x = np.arange(len(history) - k, len(history)) # Индексы эпох для последних k точек

    # Проверка на NaN или Inf в данных для регрессии
    if np.isnan(y).any() or np.isinf(y).any():
        log.warning(f"⚠️ Обнаружены NaN/Inf в истории Levenshtein за последние {k} эпох. Наклон не рассчитан.")
        return None

    try:
        # slope, intercept, r_value, p_value, std_err
        slope, _, _, _, _ = stats.linregress(x, y)
        # Проверка на NaN/Inf в результате регрессии
        if np.isnan(slope) or np.isinf(slope):
            log.warning(f"⚠️ Результат расчета наклона Левенштейна - NaN/Inf. Возвращено None.")
            return None
        return float(slope) # Убедимся, что возвращаем float
    except ValueError as ve:
         # Частая ошибка - если все y одинаковые (плато)
         if "Cannot calculate a linear regression if all x values are identical" in str(ve) or \
            "Cannot calculate a linear regression if all y values are identical" in str(ve):
             log.info(f"ℹ️ Наклон Левенштейна: достигнуто плато (все значения за k={k} эпох одинаковы). Наклон = 0.0")
             return 0.0
         else:
             log.warning(f"⚠️ Не удалось рассчитать наклон Левенштейна (ValueError): {ve}")
             return None
    except Exception as e:
        log.warning(f"⚠️ Не удалось рассчитать наклон Левенштейна (Ошибка: {type(e).__name__}): {e}")
        return None

# --- Функции analyze_errors и visualize_predictions без изменений ---
def analyze_errors(model: nn.Module, dataloader: DataLoader, idx2char: Dict[int, str], blank_idx: int, device: torch.device, top_n: int = 10):
    # ... (код без изменений) ...
    model.eval()
    errors = []
    if not dataloader.dataset or len(dataloader.dataset) == 0: # Добавил проверку len
        log.warning("⚠️ Невозможно проанализировать ошибки: валидационный датасет пуст.")
        return
    progress_bar = tqdm(dataloader, desc="🔬 Анализ ошибок", leave=False)
    with torch.no_grad():
        for batch in progress_bar:
            if not batch[0].numel(): continue
            features, targets, feature_lengths_initial, target_lengths = [b.to(device) for b in batch]
            if features.numel() == 0: continue
            try:
                logits = model(features)
                time_reduction_factor = model.get_time_reduction_factor()
                # Расчет длин как в evaluate_epoch
                feature_lengths = (feature_lengths_initial.float() / time_reduction_factor).ceil().long()
                feature_lengths = torch.clamp(feature_lengths, min=1, max=logits.shape[0]) # Важно clamp
                target_lengths = torch.clamp(target_lengths, min=1) # Важно clamp

                predictions = ctc_greedy_decode(logits, idx2char, blank_idx)
                true_labels = decode_targets(targets, target_lengths, idx2char)

                for pred, true in zip(predictions, true_labels):
                    distance = Levenshtein.distance(pred, true)
                    errors.append({"distance": distance, "prediction": pred, "true_label": true})
            except Exception as e:
                log.error(f"🛑 Ошибка во время анализа ошибок на батче: {e}")
                continue # Пропускаем батч

    errors.sort(key=lambda x: x['distance'], reverse=True)
    print(f"\n--- Топ {min(top_n, len(errors))} худших предсказаний (по расстоянию Левенштейна) ---")
    for i, error in enumerate(errors[:top_n]):
        print(f"{i+1}. Расстояние: {error['distance']}")
        print(f"   Предсказано: '{error['prediction']}'")
        print(f"   Правильно:  '{error['true_label']}'")
        print("-" * 20)

def visualize_predictions(model: nn.Module, dataloader: DataLoader, idx2char: Dict[int, str], blank_idx: int, device: torch.device, num_samples: int = 5):
    # ... (код без изменений) ...
    model.eval()
    samples_shown = 0
    if not dataloader.dataset or len(dataloader.dataset) == 0: # Добавил проверку len
        log.warning("⚠️ Невозможно визуализировать предсказания: валидационный датасет пуст.")
        return
    print(f"\n--- Примеры предсказаний ({num_samples} шт.) ---")
    with torch.no_grad():
        for batch in dataloader:
            if samples_shown >= num_samples: break
            if not batch[0].numel(): continue
            features, targets, feature_lengths_initial, target_lengths = [b.to(device) for b in batch]
            if features.numel() == 0: continue
            try:
                logits = model(features)
                # Расчет длин как в evaluate_epoch
                time_reduction_factor = model.get_time_reduction_factor()
                feature_lengths = (feature_lengths_initial.float() / time_reduction_factor).ceil().long()
                feature_lengths = torch.clamp(feature_lengths, min=1, max=logits.shape[0])
                target_lengths = torch.clamp(target_lengths, min=1)

                predictions = ctc_greedy_decode(logits, idx2char, blank_idx)
                true_labels = decode_targets(targets, target_lengths, idx2char)

                for i in range(len(predictions)):
                    if samples_shown >= num_samples: break
                    print(f"Пример {samples_shown + 1}:")
                    print(f"   Предсказано: '{predictions[i]}'")
                    print(f"   Правильно:  '{true_labels[i]}'")
                    print("-" * 20)
                    samples_shown += 1
            except Exception as e:
                 log.error(f"🛑 Ошибка во время визуализации предсказаний на батче: {e}")
                 continue # Пропускаем батч

log.info("Утилиты обновлены (добавлена функция calculate_levenshtein_slope).")

Утилиты обновлены (добавлена функция calculate_levenshtein_slope).


### Ячейка 8: Циклы Обучения и Валидации

In [18]:
# Ячейка 8: Циклы Обучения и Валидации (Русские ошибки/предупреждения) - ОБНОВЛЕНО

def train_epoch(
    model: nn.Module, dataloader: DataLoader, optimizer: optim.Optimizer, criterion: nn.CTCLoss,
    device: torch.device, scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None, use_amp: bool = True,
    scaler: Optional[torch.cuda.amp.GradScaler] = None, time_reduction_factor: float = 1.0,
    max_grad_norm: Optional[float] = None # ДОБАВЛЕН ПАРАМЕТР
) -> float:
    model.train()
    total_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(dataloader, desc="🚂 Обучение", leave=False)

    for batch in progress_bar:
        if not batch[0].numel(): continue # Пропуск пустых батчей из-за ошибок в collate/dataset
        features, targets, feature_lengths_initial, target_lengths = [b.to(device) for b in batch]
        if features.numel() == 0: continue # Дополнительная проверка

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(features) # (SeqLen, Batch, Classes)
            # Расчет длин для CTC Loss
            input_lengths = (feature_lengths_initial.float() / time_reduction_factor).ceil().long()
            # Важно: длина входа для CTC не может быть больше реальной длины выхода CNN
            input_lengths = torch.clamp(input_lengths, min=1, max=logits.shape[0])
            # Длина таргета также не может быть 0
            target_lengths = torch.clamp(target_lengths, min=1)

            # Проверка совместимости длин для CTC
            valid_indices = target_lengths <= input_lengths
            if not valid_indices.all():
                 num_invalid = (~valid_indices).sum().item()
                 # Логируем только если проблема массовая, чтобы не спамить
                 if num_invalid > len(features) * 0.1: # Например, если больше 10% батча невалидны
                     log.warning(f"⚠️ CTC Обучение: {num_invalid}/{len(features)} сэмплов имеют метку длиннее выхода CNN. Проверьте фактор сжатия.")
                 # Отфильтровываем невалидные для лосса (хотя CTC может сам их обработать как 0 лосс)
                 # log_probs = logits[:, valid_indices, :]
                 # targets = targets[valid_indices]
                 # input_lengths = input_lengths[valid_indices]
                 # target_lengths = target_lengths[valid_indices]
                 # if log_probs.numel() == 0: continue # Если весь батч отфильтрован

            # CTC требует Log Softmax
            log_probs = logits.log_softmax(dim=2)

            # Проверка на нулевые длины (хотя clamp выше должен помочь)
            if (input_lengths <= 0).any() or (target_lengths <= 0).any():
                log.error("🛑 CTC Обучение: Обнаружена нулевая или отрицательная длина входа/метки ПОСЛЕ clamp. Пропуск батча.")
                continue

            try:
                loss = criterion(log_probs, targets, input_lengths, target_lengths)
            except Exception as e:
                log.error(f"🛑 CTC Обучение: Ошибка расчета лосса: {e}. Пропуск батча.")
                continue

        # Проверка на NaN/Inf лосс
        if torch.isnan(loss) or torch.isinf(loss):
            log.warning(f"⚠️ CTC Обучение: Обнаружен NaN/Inf лосс ({loss.item()}). Пропуск батча.")
            continue # Пропускаем шаг оптимизатора

        # Backward pass и шаг оптимизатора
        if use_amp and scaler:
            scaler.scale(loss).backward()
            # Gradient Clipping ДО шага оптимизатора
            if max_grad_norm:
                scaler.unscale_(optimizer) # Нужно для клиппинга с AMP
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            # Gradient Clipping ДО шага оптимизатора
            if max_grad_norm:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        # Шаг планировщика (только для OneCycleLR здесь)
        if scheduler and isinstance(scheduler, OneCycleLR):
            scheduler.step()

        total_loss += loss.item()
        num_batches += 1
        progress_bar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{optimizer.param_groups[0]['lr']:.1E}")

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    return avg_loss

# --- Функция evaluate_epoch остается без изменений в этой итерации ---
def evaluate_epoch(
    model: nn.Module, dataloader: DataLoader, criterion: nn.CTCLoss, device: torch.device,
    idx2char: Dict[int, str], blank_idx: int, use_amp: bool = True, time_reduction_factor: float = 1.0
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_levenshtein = 0.0
    num_batches = 0
    progress_bar = tqdm(dataloader, desc="🧪 Валидация", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            if not batch[0].numel(): continue
            features, targets, feature_lengths_initial, target_lengths = [b.to(device) for b in batch]
            if features.numel() == 0: continue

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(features) # (SeqLen, Batch, Classes)
                input_lengths = (feature_lengths_initial.float() / time_reduction_factor).ceil().long()
                input_lengths = torch.clamp(input_lengths, min=1, max=logits.shape[0])
                target_lengths = torch.clamp(target_lengths, min=1)

                # В валидации просто логируем, если есть проблемы, но считаем лосс/метрики
                valid_indices = target_lengths <= input_lengths
                if not valid_indices.all():
                    num_invalid = (~valid_indices).sum().item()
                    if num_invalid > len(features) * 0.1:
                         log.warning(f"⚠️ CTC Валидация: {num_invalid}/{len(features)} сэмплов имеют метку длиннее выхода CNN.")

                if (input_lengths <= 0).any() or (target_lengths <= 0).any():
                    log.error("🛑 CTC Валидация: Нулевая или отрицательная длина входа/метки ПОСЛЕ clamp.")
                    continue # Пропускаем расчет лосса/метрик для этого батча

                log_probs = logits.log_softmax(dim=2)
                try:
                    # Считаем лосс только по валидным (хотя CTC должен справиться)
                    loss = criterion(log_probs, targets, input_lengths, target_lengths)
                except Exception as e:
                    log.error(f"🛑 CTC Валидация: Ошибка расчета лосса: {e}. Пропуск батча.")
                    continue

            if not (torch.isnan(loss) or torch.isinf(loss)):
                total_loss += loss.item()
            else:
                log.warning(f"⚠️ CTC Валидация: Обнаружен NaN/Inf лосс ({loss.item()}).")

            # Декодируем и считаем Левенштейн для ВСЕГО батча
            predictions = ctc_greedy_decode(logits, idx2char, blank_idx)
            true_labels = decode_targets(targets, target_lengths, idx2char)
            batch_levenshtein = calculate_levenshtein(predictions, true_labels)

            total_levenshtein += batch_levenshtein # Суммируем среднее по батчу
            num_batches += 1
            progress_bar.set_postfix(loss=f"{loss.item():.4f}", lev=f"{batch_levenshtein:.3f}")

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    avg_levenshtein = total_levenshtein / num_batches if num_batches > 0 else float('inf') # Усредняем средние по батчам
    return avg_loss, avg_levenshtein

log.info("Функция train_epoch обновлена (добавлен max_grad_norm).")

Функция train_epoch обновлена (добавлен max_grad_norm).


### Ячейка 9: Основная Функция Обучения (run_training)

In [19]:
# --- Ячейка 9: Основная Функция Обучения (run_training - Расчет ФИНАЛЬНОГО наклона) ---

# Импорты (убедитесь, что все нужные импорты есть)
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import _LRScheduler, ReduceLROnPlateau, OneCycleLR
from torch.cuda.amp import GradScaler
from pathlib import Path
import time
import numpy as np
from scipy import stats
from omegaconf import DictConfig, OmegaConf
import optuna
from typing import Dict, Any, Optional, Tuple, List
import logging
# log = logging.getLogger(__name__) # Должен быть настроен ранее
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Должен быть определен ранее

# Функции train_epoch, evaluate_epoch, calculate_levenshtein_slope, log_model_summary
# должны быть определены в предыдущих ячейках.

# --- Определение log_model_summary (без изменений по сравнению с прошлым разом) ---
def log_model_summary(model: nn.Module, cfg: DictConfig):
    # ... (код функции log_model_summary) ...
    log.info("--- 🧠 Конфигурация Модели 🧠 ---")
    m_cfg = cfg.model_params
    try:
        log.info(f" > Активация: {m_cfg.activation_type}")
        log.info(f" > Раздельная обработка каналов (groups=2 в 1м блоке): {'Да' if m_cfg.process_channels_separately else 'Нет'}")
        log.info(f" > ResBlocks ({len(m_cfg.block_out_channels)} стадии):")
        num_stages = len(m_cfg.block_out_channels)
        for i in range(num_stages):
             k_size = getattr(m_cfg, f'block_kernel_size_{i}', 'N/A')
             stride = getattr(m_cfg, f'block_stride_{i}', 'N/A')
             log.info(f"    Стадия {i+1}: Каналы={m_cfg.block_out_channels[i]}, Ядро={k_size}, Stride={stride}")
        log.info(f" > RNN:")
        log.info(f"    Тип={m_cfg.rnn_type}, Скрытый размер={m_cfg.rnn_hidden_size}, Слои={m_cfg.rnn_num_layers}, Dropout={m_cfg.rnn_dropout}")
        log.info(f" > Классификатор:")
        log.info(f"    Dropout={m_cfg.classifier_dropout}, Размер словаря={m_cfg.vocab_size}")
        if hasattr(model, 'get_time_reduction_factor'):
            reduction_factor = model.get_time_reduction_factor()
            log.info(f" > 📉 Фактор сжатия времени CNN: {reduction_factor:.2f}x")
        else: log.warning(" > ⚠️ Не удалось получить фактор сжатия из модели.")
        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info(f" > ⚙️ Всего обучаемых параметров: {num_params:,}")
    except AttributeError as e: log.warning(f"⚠️ Не удалось вывести полную информацию о модели: отсутствует атрибут конфига: {e}")
    except Exception as e: log.error(f"🛑 Ошибка при логировании информации о модели: {e}")
    log.info("---------------------------------")
# --- Конец определения log_model_summary ---

# --- Определение run_training (с изменениями) ---
def run_training(
    cfg: DictConfig, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, criterion: nn.CTCLoss,
    optimizer: optim.Optimizer, scheduler: Optional[_LRScheduler], device: torch.device,
    idx2char: Dict[int, str], blank_idx: int, output_dir: Path, trial: Optional[optuna.Trial] = None
) -> Dict[str, Any]:
    log.info(f"--- ▶️ Старт цикла обучения (Папка: {output_dir.name}) ---")
    output_dir.mkdir(parents=True, exist_ok=True)
    log_model_summary(model, cfg)

    scaler = GradScaler() if cfg.train_params.use_amp and device.type == 'cuda' else None
    use_amp_effective = cfg.train_params.use_amp and device.type == 'cuda'
    if use_amp_effective: log.info("⚡ Включено Automatic Mixed Precision (AMP).")
    else: log.info(" AMP отключено.")

    max_grad_norm = cfg.train_params.max_grad_norm
    if max_grad_norm: log.info(f"✂️ Включен Gradient Clipping (max_norm={max_grad_norm}).")

    best_val_levenshtein = float('inf')
    epochs_no_improve = 0
    best_model_path = output_dir / "best_model.pth"
    # Убрали 'lev_slope' из инициализации истории, т.к. считаем только в конце
    history = {'train_loss': [], 'val_loss': [], 'val_levenshtein': [], 'lr': []}
    num_epochs = cfg.train_params.num_epochs

    log.info(f"🚀 Начинаем обучение ({num_epochs} эпох)...")
    time_reduction_factor = 1.0
    if hasattr(model, 'get_time_reduction_factor'):
        time_reduction_factor = model.get_time_reduction_factor()
    else: log.warning("⚠️ model.get_time_reduction_factor() не найден! Используется time_reduction_factor = 1.0")

    # --- Цикл обучения ---
    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # Обучение
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, device,
            scheduler if isinstance(scheduler, OneCycleLR) else None,
            use_amp_effective, scaler, time_reduction_factor, max_grad_norm
        )
        history['train_loss'].append(train_loss)

        # Валидация
        val_loss, val_levenshtein = evaluate_epoch(
            model, val_loader, criterion, device, idx2char, blank_idx,
            use_amp_effective, time_reduction_factor
        )
        history['val_loss'].append(val_loss)
        history['val_levenshtein'].append(val_levenshtein)

        current_lr = optimizer.param_groups[0]['lr']
        history['lr'].append(current_lr)

        # --- УБРАН РАСЧЕТ НАКЛОНА ИЗНУТРИ ЦИКЛА ---

        # Логирование эпохи (без наклона)
        epoch_duration = time.time() - epoch_start_time
        epoch_summary = f"📊 Эпоха [{epoch+1:>{len(str(num_epochs))}}/{num_epochs}] | " \
                        f"T: {epoch_duration:.1f}s | " \
                        f"Loss(T/V): {train_loss:.4f}/{val_loss:.4f} | " \
                        f"Lev(V): {val_levenshtein:.4f} | LR: {current_lr:.1E}"
        log.info(epoch_summary)

        # Шаг планировщика
        if scheduler and isinstance(scheduler, ReduceLROnPlateau):
            if val_levenshtein is not None and not (np.isnan(val_levenshtein) or np.isinf(val_levenshtein)):
                 scheduler.step(val_levenshtein)
                 new_lr = optimizer.param_groups[0]['lr']
                 if new_lr < current_lr: log.info(f"  📉 LR снижен планировщиком до {new_lr:.1E}")
            else: log.warning(f"  ⚠️ Пропуск шага scheduler.step() из-за невалидного Levenshtein ({val_levenshtein})")

        # Сохранение лучшей модели
        save_model = False
        # ... (логика сохранения без изменений) ...
        if val_levenshtein is None or np.isnan(val_levenshtein) or np.isinf(val_levenshtein):
             log.warning(f"  ⚠️ Получено невалидное значение Levenshtein ({val_levenshtein}). Модель не будет сохранена как лучшая.")
        elif val_levenshtein < best_val_levenshtein:
            improvement = best_val_levenshtein - val_levenshtein
            log.info(f"  ✅ Улучшение Levenshtein ({best_val_levenshtein:.4f} -> {val_levenshtein:.4f}, Δ={improvement:.4f})! Сохраняем модель -> {best_model_path.name}")
            best_val_levenshtein = val_levenshtein
            save_model = True
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            log.info(f"  ⏳ Без улучшения Levenshtein ({epochs_no_improve}/{cfg.train_params.early_stopping_patience})")

        if save_model:
            try:
                # ... (код сохранения чекпоинта) ...
                torch.save({
                    'epoch': epoch + 1, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
                    'best_val_levenshtein': best_val_levenshtein,
                    'config': OmegaConf.to_container(cfg, resolve=True), 'idx2char': idx2char
                }, best_model_path)
            except Exception as e: log.error(f"🛑 Ошибка сохранения чекпоинта: {e}")


        # Optuna Pruning & Reporting (только основная метрика внутри цикла)
        if trial:
            if val_levenshtein is not None and not (np.isnan(val_levenshtein) or np.isinf(val_levenshtein)):
                trial.report(val_levenshtein, epoch)
            # --- УБРАН trial.set_user_attr для наклона ВНУТРИ ЦИКЛА ---
            if trial.should_prune():
                log.warning(f"⚠️ Trial {trial.number} остановлен Pruner'ом Optuna на эпохе {epoch+1}.")
                # ... (код сохранения перед прунингом) ...
                raise optuna.TrialPruned()

        # Ранняя остановка
        if epochs_no_improve >= cfg.train_params.early_stopping_patience:
            log.info(f"⏹️ Ранняя остановка после {epoch+1} эпох без улучшения.")
            break
    # --- Конец цикла обучения ---

    log.info(f"--- 🏁 Конец цикла обучения ---")

    # --- ДОБАВЛЕНО: Расчет ФИНАЛЬНОГО наклона Levenshtein ---
    final_slope = None
    final_slope_window_used = None
    # Фактическое количество завершенных эпох
    completed_epochs = len(history['val_levenshtein'])
    slope_percentage = cfg.train_params.get('levenshtein_slope_percentage', 0.4)
    slope_func_exists = 'calculate_levenshtein_slope' in globals() and callable(globals()['calculate_levenshtein_slope'])

    if completed_epochs >= 2 and slope_func_exists: # Нужно хотя бы 2 точки для наклона
        # Рассчитываем окно k как % от ФАКТИЧЕСКИ завершенных эпох
        final_slope_window_used = max(2, int(np.ceil(completed_epochs * slope_percentage)))
        # Окно не может быть больше числа доступных точек
        final_slope_window_used = min(final_slope_window_used, completed_epochs)

        # Берем только валидные значения из истории
        valid_history = [lv for lv in history['val_levenshtein'] if lv is not None and not (np.isnan(lv) or np.isinf(lv))]

        # Проверяем, достаточно ли ВАЛИДНЫХ точек для рассчитанного окна
        if len(valid_history) >= final_slope_window_used:
            # Рассчитываем наклон по ПОСЛЕДНИМ k точкам ВАЛИДНОЙ истории
            final_slope = calculate_levenshtein_slope(valid_history, k=final_slope_window_used)
            log.info(f"📈 Рассчитан финальный наклон Levenshtein:")
            log.info(f"   - Использовано эпох: {completed_epochs}")
            log.info(f"   - Процент для окна: {slope_percentage*100:.0f}%")
            log.info(f"   - Расчетное окно (k): {final_slope_window_used}")
            log.info(f"   - Значение наклона: {f'{final_slope:.3E}' if final_slope is not None else 'N/A'}")
        else:
            log.warning(f"⚠️ Недостаточно валидных точек ({len(valid_history)}) для расчета финального наклона с окном {final_slope_window_used}. Пропуск.")
            final_slope_window_used = None # Сбрасываем окно, если расчет не удался
    elif not slope_func_exists:
         log.warning("⚠️ Функция calculate_levenshtein_slope не найдена. Финальный наклон не рассчитан.")
    elif completed_epochs < 2:
         log.info(f"ℹ️ Пройдено менее 2 эпох ({completed_epochs}). Финальный наклон не рассчитывается.")
    # --- КОНЕЦ РАСЧЕТА ФИНАЛЬНОГО НАКЛОНА ---


    # Загрузка лучшей модели
    # ... (код загрузки модели без изменений) ...
    loaded_model = None
    loaded_idx2char = idx2char
    if best_model_path.exists():
        log.info(f"💾 Загрузка лучшей модели из: {best_model_path.name}")
        try:
            checkpoint = torch.load(best_model_path, map_location=device)
            saved_cfg_dict = checkpoint.get('config')
            if saved_cfg_dict:
                saved_cfg = OmegaConf.create(saved_cfg_dict)
                if 'vocab_size' not in saved_cfg.model_params or saved_cfg.model_params.vocab_size is None:
                    log.warning("⚠️ Vocab size не найден в сохраненном конфиге чекпоинта. Используется текущий.")
                    saved_cfg.model_params.vocab_size = cfg.model_params.vocab_size
                loaded_model = MorseRecognizer1D(saved_cfg).to(device)
                missing, unexpected = loaded_model.load_state_dict(checkpoint['model_state_dict'], strict=False)
                if missing: log.warning(f"  ⚠️ Пропущенные ключи при загрузке state_dict: {missing}")
                if unexpected: log.warning(f"  ⚠️ Лишние ключи при загрузке state_dict: {unexpected}")
                loaded_idx2char = checkpoint.get('idx2char', idx2char)
                log.info("  ✅ Лучшая модель успешно загружена.")
            else:
                 log.error("🛑 Конфигурация не найдена в чекпоинте. Невозможно загрузить модель.")
                 loaded_model = model
        except Exception as e:
            log.error(f"🛑 Ошибка загрузки лучшей модели: {e}. Возвращается модель последней эпохи.")
            loaded_model = model
    else:
        log.warning(f"⚠️ Лучшая модель не найдена по пути ({best_model_path}). Возвращается модель последней эпохи.")
        loaded_model = model

    # Формирование результатов
    results = {
        "best_val_levenshtein": best_val_levenshtein if best_val_levenshtein != float('inf') else None,
        "best_model_path": str(best_model_path) if best_model_path.exists() else None,
        "history": history,
        "final_model": loaded_model,
        "idx2char": loaded_idx2char,
        # Добавляем финальный наклон и окно, использованное для его расчета
        "final_levenshtein_slope": final_slope,
        "levenshtein_slope_window_used": final_slope_window_used
    }
    # Финальный лог теперь не дублирует информацию о наклоне, она была выше
    # log.info(f" > Финальный наклон Levenshtein (k={final_slope_window_used if final_slope_window_used is not None else 'N/A'}): {f'{final_slope:.3E}' if final_slope is not None else 'N/A'}")

    return results
# --- Конец определения run_training ---

log.info("Функция run_training обновлена (расчет только финального наклона Levenshtein после цикла).")

Функция run_training обновлена (расчет только финального наклона Levenshtein после цикла).


### Ячейка 10: Основная Логика Запуска с Hydra (@hydra.main)

In [20]:
# Ячейка 10: Основная Логика Запуска с Hydra (@hydra.main) - ИСПРАВЛЕНО (Path handling)

import datetime
import traceback
import pprint # Для красивого вывода словарей

# --- Функция Objective для Optuna HPO (Читает distributions из cfg) ---
def objective(trial: optuna.Trial, base_cfg: DictConfig) -> float:
    # ... (начало функции objective без изменений) ...
    cfg = base_cfg.copy()
    OmegaConf.set_struct(cfg, False)

    log.info(f"\n--- 📊 Старт Optuna Trial #{trial.number} 📊 ---")

    # --- Применение гиперпараметров из cfg.hpo_params.param_distributions ---
    hpo_dist = cfg.hpo_params.param_distributions
    suggested_params = {}
    if not hpo_dist:
        log.warning("⚠️ HPO: Словарь param_distributions пуст в конфигурации!")
    else:
        log.info(f" > ⚙️ Подбор параметров для Trial #{trial.number}:")
        for param_path, dist_config in hpo_dist.items():
            config = OmegaConf.to_container(dist_config, resolve=True)
            suggest_type = config.pop("type", None)
            if not suggest_type:
                log.error(f"🛑 HPO Error: 'type' не указан для параметра '{param_path}' в param_distributions.")
                continue
            try:
                suggested_value = None
                # ... (код подбора: suggest_categorical, suggest_float, suggest_int) ...
                if suggest_type == "categorical":
                    suggested_value = trial.suggest_categorical(param_path, **config)
                elif suggest_type == "float":
                    if "min" in config: config["low"] = config.pop("min")
                    if "max" in config: config["high"] = config.pop("max")
                    suggested_value = trial.suggest_float(param_path, **config)
                elif suggest_type == "int":
                    if "min" in config: config["low"] = config.pop("min")
                    if "max" in config: config["high"] = config.pop("max")
                    suggested_value = trial.suggest_int(param_path, **config)
                else:
                    log.error(f"🛑 HPO Error: Неподдерживаемый тип '{suggest_type}' для параметра '{param_path}'.")
                    continue

                OmegaConf.update(cfg, param_path, suggested_value)
                log_value = f"{suggested_value:.1E}" if isinstance(suggested_value, float) and config.get("log") else suggested_value
                suggested_params[param_path] = log_value
            except Exception as e:
                log.error(f"🛑 HPO Error: Ошибка при предложении параметра '{param_path}': {e}")
                raise optuna.TrialPruned(f"Ошибка конфигурации HPO для {param_path}")

        log.info("   Предложенные параметры:")
        log.info(pprint.pformat(suggested_params, indent=4, width=80))

    OmegaConf.set_struct(cfg, True)

    # --- Подготовка данных (без изменений) ---
    # ... (код подготовки данных) ...
    log.info(f" > 💾 Подготовка данных...")
    current_seed = cfg.data_params.random_seed
    set_seed(current_seed + trial.number)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    metadata_path = Path(cfg.paths.data_dir)
    feature_path = Path(cfg.paths.feature_dir)
    try:
        df_full = load_metadata(metadata_path, cfg)
        data_fraction = cfg.data_params.dataset_fraction
        if not (0.0 < data_fraction <= 1.0): data_fraction = 1.0
        if data_fraction < 1.0:
            df = df_full.sample(frac=data_fraction, random_state=current_seed + trial.number) # Используем seed trial'а
        else:
            df = df_full
        log.info(f"   Используется {len(df)}/{len(df_full)} сэмплов (доля: {data_fraction:.2f}).")
        char2idx, idx2char, vocab_size = create_vocabulary(df, cfg)
        OmegaConf.set_struct(cfg, False) # Временно разрешаем изменение
        cfg.model_params.vocab_size = vocab_size
        OmegaConf.set_struct(cfg, True) # Снова блокируем
        blank_idx = char2idx[cfg.data_params.blank_char]
        train_df, val_df = split_data(df, cfg.data_params.train_val_split_ratio, current_seed + trial.number) # Используем seed trial'а
        train_dataset = MorseDataset(train_df, feature_path, char2idx, cfg)
        val_dataset = MorseDataset(val_df, feature_path, char2idx, cfg)
        collate_fn_with_blank = lambda batch: collate_fn(batch, blank_idx)
        if len(train_dataset) == 0 or len(val_dataset) == 0:
            log.error(f"🛑 Trial #{trial.number}: Пустой датасет после фильтрации/сэмплинга. Pruning.")
            raise optuna.TrialPruned("Пустой датасет")
        train_loader = DataLoader(train_dataset, batch_size=cfg.train_params.batch_size, shuffle=True, collate_fn=collate_fn_with_blank, num_workers=0) # num_workers=0
        val_loader = DataLoader(val_dataset, batch_size=cfg.train_params.batch_size, shuffle=False, collate_fn=collate_fn_with_blank, num_workers=0) # num_workers=0
        log.info(f"   Загрузчики данных созданы (Обуч батчей: {len(train_loader)}, Вал батчей: {len(val_loader)}).")
    except (FileNotFoundError, KeyError, ValueError) as e:
        log.error(f"🛑 Trial #{trial.number}: Ошибка загрузки данных: {e}. Pruning.")
        raise optuna.TrialPruned(f"Ошибка данных: {e}")
    except Exception as e:
        log.error(f"🛑 Trial #{trial.number}: Неожиданная ошибка загрузки данных: {e}", exc_info=True)
        raise optuna.TrialPruned(f"Ошибка данных: {e}")

    # --- Инициализация модели и оптимизатора (без изменений) ---
    # ... (код инициализации) ...
    log.info(f" > 🚀 Инициализация модели и оптимизатора...")
    try:
        model = MorseRecognizer1D(cfg).to(device)
        criterion = nn.CTCLoss(blank=blank_idx, zero_infinity=True)
        optimizer_class = getattr(optim, cfg.train_params.optimizer)
        optimizer = optimizer_class(model.parameters(), lr=cfg.train_params.learning_rate, weight_decay=cfg.train_params.weight_decay)
        scheduler = None
        steps_per_epoch = len(train_loader) if len(train_loader) > 0 else 1
        if cfg.train_params.scheduler == 'OneCycleLR':
            scheduler = OneCycleLR(optimizer, max_lr=cfg.train_params.max_lr, epochs=cfg.train_params.num_epochs, steps_per_epoch=steps_per_epoch, pct_start=0.15, div_factor=5, final_div_factor=20)
        elif cfg.train_params.scheduler == 'ReduceLROnPlateau':
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=cfg.train_params.scheduler_reduce_factor, patience=cfg.train_params.scheduler_patience, verbose=False)
        log.info(f"   Инициализация завершена.")
    except Exception as e:
        log.error(f"🛑 Trial #{trial.number}: Ошибка инициализации: {e}", exc_info=True)
        raise optuna.TrialPruned(f"Ошибка инициализации: {e}")

     # --- Запуск обучения ---
    # Создаем директорию для артефактов этого trial'а внутри общей папки HPO
    base_hpo_artifacts_dir = Path(f"hpo_run_{cfg.run_timestamp}") # Используем общий timestamp запуска
    trial_output_dir = base_hpo_artifacts_dir / f"trial_{trial.number}"
    try:
        # Запускаем обучение
        results = run_training(
            cfg=cfg, model=model, train_loader=train_loader, val_loader=val_loader,
            criterion=criterion, optimizer=optimizer, scheduler=scheduler, device=device,
            idx2char=idx2char, blank_idx=blank_idx, output_dir=trial_output_dir, trial=trial
        )

        # --- ИЗМЕНЕНО: Получаем и сохраняем финальный наклон ---
        # Получаем основную метрику и финальный наклон из результатов
        metric_value = results.get('best_val_levenshtein') # Может быть None
        final_slope = results.get('final_levenshtein_slope') # Может быть None
        final_slope_window = results.get('levenshtein_slope_window_used') # Может быть None

        # Обрабатываем случай, если метрика None (например, обучение не улучшилось)
        # Optuna ожидает float, поэтому возвращаем худшее значение
        if metric_value is None:
             metric_value_for_optuna = float('inf') if cfg.hpo_params.direction == "minimize" else float('-inf')
             log.warning(f"⚠️ Trial #{trial.number}: best_val_levenshtein is None. Returning {metric_value_for_optuna} to Optuna.")
        else:
             metric_value_for_optuna = metric_value

        log.info(f"--- ✅ Завершен Optuna Trial #{trial.number} | "
                 f"Лучший Levenshtein: {metric_value:.4f} | "
                 f"Фин. наклон (k={final_slope_window if final_slope_window is not None else 'N/A'}): {f'{final_slope:.3E}' if final_slope is not None else 'N/A'} ---")

        # Сохраняем основную метрику как user_attr (для удобства)
        if metric_value is not None:
             trial.set_user_attr("best_val_levenshtein", metric_value)

        # Сохраняем ФИНАЛЬНЫЙ наклон и окно в атрибуты Optuna, если они были рассчитаны
        if final_slope is not None:
            trial.set_user_attr("final_levenshtein_slope", final_slope)
        if final_slope_window is not None:
            trial.set_user_attr("final_slope_window", final_slope_window)
        # --- КОНЕЦ ИЗМЕНЕНИЙ ---

        # Сохраняем путь к лучшей модели
        best_model_path_str = results.get('best_model_path')
        if best_model_path_str:
            try:
                # Преобразуем в относительный путь для переносимости
                rel_path = Path(best_model_path_str).relative_to(Path.cwd())
                rel_path_posix = rel_path.as_posix()
                trial.set_user_attr("best_model_relative_path", rel_path_posix)
                log.info(f"   -> Сохранен атрибут best_model_relative_path: {rel_path_posix}")
            except ValueError: # Если пути на разных дисках (маловероятно здесь)
                 trial.set_user_attr("best_model_absolute_path", best_model_path_str)
                 log.warning(f"   -> Не удалось получить относительный путь. Сохранен абсолютный: {best_model_path_str}")


        # Возвращаем основную метрику для оптимизации Optuna
        return metric_value_for_optuna

    except optuna.TrialPruned as e:
        log.warning(f"--- ⏹️ Прерван Optuna Trial #{trial.number} (Pruned: {e}) ---")
        raise # Передаем исключение дальше, чтобы Optuna его обработал
    except Exception as e:
        log.error(f"🛑 Trial #{trial.number}: Неожиданная ошибка во время выполнения trial'а: {e}", exc_info=True)
        # Помечаем trial как проваленный и сохраняем ошибку
        trial.set_user_attr("status", "FailedDuringTrial")
        trial.set_user_attr("error_message", f"{type(e).__name__}: {e}")
        try: # Попытка записать лог ошибки
            trial_output_dir.mkdir(parents=True, exist_ok=True)
            with open(trial_output_dir / "error_log.txt", "w", encoding='utf-8') as f:
                f.write(f"{datetime.datetime.now()}\nError in Trial {trial.number}:\n{e}\n")
                f.write(traceback.format_exc())
        except Exception as log_e: log.error(f" > Не удалось записать лог ошибки для trial {trial.number}: {log_e}")
        log.info(f"--- ❌ Провален Optuna Trial #{trial.number} (ошибка во время выполнения) ---")
        # Возвращаем худшее возможное значение, чтобы Optuna не считал этот trial успешным
        return float('inf') if cfg.hpo_params.direction == "minimize" else float('-inf')


# --- Основная функция main_runner (без изменений в логике HPO) ---
def main_runner(cfg: DictConfig) -> Optional[float]:
    OmegaConf.set_struct(cfg, False)
    run_timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    cfg.run_timestamp = run_timestamp # Добавляем общий timestamp для папки HPO
    OmegaConf.set_struct(cfg, True)

    log.info("========================================================")
    log.info(f"🚀 Старт Запуска: {run_timestamp}")
    run_mode = "HPO (Подбор гиперпараметров)" if cfg.enable_hpo else "Одиночный запуск (Обучение/Дообучение)"
    log.info(f" Режим: {run_mode}")
    # Папка теперь использует общий timestamp
    output_root = Path(f"{'hpo_run' if cfg.enable_hpo else 'single_run'}_{run_timestamp}")
    log.info(f" Корневая папка вывода: {output_root.name}")
    log.info("========================================================")

    set_seed(cfg.data_params.random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"💻 Устройство: {device}")

    db_dir = Path("optuna_databases")
    db_dir.mkdir(parents=True, exist_ok=True)

    if cfg.enable_hpo:
        # --- Режим HPO ---
        log.info("--- 🔍 Настройка Optuna HPO ---")
        hpo_cfg = cfg.hpo_params
        storage_name_for_optuna = None
        if hpo_cfg.storage_name:
            # Обработка пути к БД (централизованное хранилище)
            if hpo_cfg.storage_name.startswith("sqlite:///"):
                db_file_part = hpo_cfg.storage_name.replace("sqlite:///", "")
                db_path = Path(db_file_part)
                # Если путь относительный, делаем его относительно CWD
                if not db_path.is_absolute():
                    db_path = Path.cwd() / db_path
                final_db_path = db_path.resolve()
            else: # Если просто имя файла, кладем в папку optuna_databases
                final_db_path = (db_dir / hpo_cfg.storage_name).resolve()

            try: # Создаем родительскую папку, если нужно
                final_db_path.parent.mkdir(parents=True, exist_ok=True)
                storage_name_for_optuna = f"sqlite:///{final_db_path}"
                log.info(f" > Хранилище (Storage): {storage_name_for_optuna}")
            except Exception as e:
                log.error(f"🛑 Не удалось подготовить путь к БД '{final_db_path}': {e}. HPO невозможен.")
                return None
        else:
            log.warning(f"⚠️ hpo_params.storage_name не задан. Optuna будет использовать in-memory storage.")

        # Имя исследования теперь уникально для каждого запуска main_runner
        study_name = hpo_cfg.study_name or f"morse_hpo_{run_timestamp}"
        log.info(f" > Имя исследования (Study): '{study_name}'")

        try:
            study = optuna.create_study(
                storage=storage_name_for_optuna,
                study_name=study_name,
                direction=hpo_cfg.direction,
                load_if_exists=False # Важно: Не загружать, если существует, а создавать новое уникальное
            )
            log.info(f" > Исследование '{study_name}' создано.")
        except Exception as e:
            log.error(f"🛑 Не удалось создать исследование Optuna '{study_name}': {e}", exc_info=True)
            return None

        log.info(f" > Запуск {hpo_cfg.n_trials} trials...")
        try:
            # Передаем КОПИЮ конфига в objective, чтобы изменения не влияли на другие trial'ы
            study.optimize(lambda trial: objective(trial, cfg.copy()),
                           n_trials=hpo_cfg.n_trials,
                           gc_after_trial=True if device.type == 'cuda' else False,
                           show_progress_bar=True) # Можно включить прогресс-бар
        except KeyboardInterrupt:
            log.warning("\n--- ⏹️ HPO прерван пользователем ---")
        except Exception as e:
            log.error(f"\n--- 🛑 Непредвиденная ошибка оптимизации Optuna: {e} ---", exc_info=True)

        log.info("--- 🏁 HPO Завершен ---")
        # Анализ результатов HPO
        log.info(f" > Исследование: '{study.study_name}'")
        completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        best_trial_value = None
        if completed_trials:
             try:
                 best_trial = study.best_trial
                 log.info(f" > 🏆 Лучший Trial #{best_trial.number}: {hpo_cfg.metric_to_optimize} = {best_trial.value:.4f}")
                 # Выводим атрибуты лучшего trial'а, включая наклон
                 log.info("   Атрибуты лучшего trial'а:")
                 log.info(pprint.pformat(best_trial.user_attrs, indent=4, width=100))
                 log.info("   Лучшие параметры:")
                 log.info(pprint.pformat(best_trial.params, indent=4, width=80))
                 best_trial_value = best_trial.value
             except ValueError: log.warning("⚠️ Не удалось определить лучший HPO trial.")
             except Exception as e: log.error(f"🛑 Ошибка получения лучшего HPO trial: {e}")

             try: # Сохраняем DataFrame и графики Optuna
                 df_results = study.trials_dataframe(attrs=("number", "value", "params", "state", "user_attrs", "datetime_start", "duration"))
                 results_path = output_root / "hpo_results_summary.csv"
                 df_results.to_csv(results_path, index=False)
                 log.info(f" > 💾 Сводка HPO сохранена: {results_path.name}")

                 # Сохраняем графики Optuna (требует plotly)
                 try:
                     fig_history = optuna.visualization.plot_optimization_history(study)
                     fig_history.write_image(output_root / "hpo_optimization_history.png")
                     fig_importance = optuna.visualization.plot_param_importances(study)
                     fig_importance.write_image(output_root / "hpo_param_importances.png")
                     log.info(f" > 📈 Графики HPO сохранены в: {output_root.name}")
                 except ImportError: log.warning("⚠️ Plotly не найден. Пропустите сохранение графиков Optuna. Установите: pip install plotly")
                 except Exception as plot_e: log.error(f"🛑 Не удалось сохранить графики Optuna: {plot_e}")

             except Exception as e: log.error(f"🛑 Не удалось сохранить сводку HPO: {e}")
        else: log.info(" > Нет успешно завершенных HPO trials.")
        return best_trial_value

    else:
        # --- Режим Обычного Обучения / Дообучения ---
        # ... (код одиночного запуска без изменений) ...
        log.info("--- ⚙️ Режим: Одиночный запуск ---")
        output_dir = output_root # Папка для этого запуска
        output_dir.mkdir(parents=True, exist_ok=True)
        log.info(f" > Папка вывода: {output_dir}")
        try:
            config_save_path = output_dir / "config_used.yaml"
            OmegaConf.save(cfg, config_save_path)
            log.info(f" > 💾 Конфигурация сохранена: {config_save_path.name}")
        except Exception as e:
            log.error(f"🛑 Не удалось сохранить конфигурацию: {e}")

        log.info(" > 💾 Подготовка данных...")
        try:
            metadata_path = Path(cfg.paths.data_dir)
            feature_path = Path(cfg.paths.feature_dir)
            df_full = load_metadata(metadata_path, cfg)
            data_fraction = cfg.data_params.dataset_fraction
            if not (0.0 < data_fraction <= 1.0): data_fraction = 1.0
            if data_fraction < 1.0:
                df = df_full.sample(frac=data_fraction, random_state=cfg.data_params.random_seed)
                log.info(f"   Используется {data_fraction*100:.1f}% данных ({len(df)} сэмплов).")
            else:
                df = df_full
                log.info(f"   Используется полный датасет ({len(df)} сэмплов).")

            char2idx, idx2char, vocab_size = create_vocabulary(df, cfg)
            OmegaConf.set_struct(cfg, False)
            cfg.model_params.vocab_size = vocab_size
            OmegaConf.set_struct(cfg, True)
            blank_idx = char2idx[cfg.data_params.blank_char]
            train_df, val_df = split_data(df, cfg.data_params.train_val_split_ratio, cfg.data_params.random_seed)
            train_dataset = MorseDataset(train_df, feature_path, char2idx, cfg)
            val_dataset = MorseDataset(val_df, feature_path, char2idx, cfg)
            if len(train_dataset) == 0 or len(val_dataset) == 0:
                log.error("🛑 Пустой датасет после фильтрации/сэмплинга. Прерывание.")
                return None
            collate_fn_with_blank = lambda batch: collate_fn(batch, blank_idx)
            train_loader = DataLoader(train_dataset, batch_size=cfg.train_params.batch_size, shuffle=True, collate_fn=collate_fn_with_blank, num_workers=0)
            val_loader = DataLoader(val_dataset, batch_size=cfg.train_params.batch_size, shuffle=False, collate_fn=collate_fn_with_blank, num_workers=0)
            log.info(f"   Загрузчики данных созданы (Обуч батчей: {len(train_loader)}, Вал батчей: {len(val_loader)}).")
            log.info(f"   Размер словаря: {vocab_size}")
        except (FileNotFoundError, KeyError, ValueError) as e:
            log.error(f"🛑 Ошибка загрузки данных: {e}. Прерывание.", exc_info=False); return None
        except Exception as e:
            log.error(f"🛑 Неожиданная ошибка загрузки данных: {e}. Прерывание.", exc_info=True); return None

        log.info(" > 🚀 Инициализация модели...")
        try: model = MorseRecognizer1D(cfg).to(device)
        except Exception as e: log.error(f"🛑 Ошибка инициализации модели: {e}. Прерывание.", exc_info=True); return None

        if cfg.finetune_checkpoint_path:
            checkpoint_path = Path(cfg.finetune_checkpoint_path).resolve()
            log.info(f" > ⬇️ Попытка дообучения из: {checkpoint_path.name}")
            if checkpoint_path.exists():
                try:
                    checkpoint = torch.load(checkpoint_path, map_location=device)
                    if 'idx2char' in checkpoint:
                        saved_idx2char = checkpoint['idx2char']
                        if len(saved_idx2char) != vocab_size:
                             log.warning(f"  ⚠️ Несовпадение размера словаря! Чекпоинт: {len(saved_idx2char)}, Текущий: {vocab_size}.")
                    missing, unexpected = model.load_state_dict(checkpoint['model_state_dict'], strict=False)
                    if missing: log.warning(f"  ⚠️ Пропущенные ключи при загрузке весов: {missing}")
                    if unexpected: log.warning(f"  ⚠️ Лишние ключи при загрузке весов: {unexpected}")
                    log.info("  ✅ Веса из чекпоинта загружены (strict=False).")
                except Exception as e:
                    log.error(f"  🛑 Ошибка загрузки чекпоинта: {e}. Обучение с нуля.", exc_info=False)
            else:
                log.warning(f"  ⚠️ Чекпоинт для дообучения не найден: {checkpoint_path}. Обучение с нуля.")

        log.info(" > ⚙️ Инициализация оптимизатора и планировщика...")
        try:
            criterion = nn.CTCLoss(blank=blank_idx, zero_infinity=True)
            optimizer_class = getattr(optim, cfg.train_params.optimizer)
            optimizer = optimizer_class(model.parameters(), lr=cfg.train_params.learning_rate, weight_decay=cfg.train_params.weight_decay)
            scheduler = None
            steps_per_epoch = len(train_loader) if len(train_loader) > 0 else 1
            if cfg.train_params.scheduler == 'OneCycleLR':
                scheduler = OneCycleLR(optimizer, max_lr=cfg.train_params.max_lr, epochs=cfg.train_params.num_epochs, steps_per_epoch=steps_per_epoch)
            elif cfg.train_params.scheduler == 'ReduceLROnPlateau':
                scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=cfg.train_params.scheduler_reduce_factor, patience=cfg.train_params.scheduler_patience, verbose=False)
            log.info(f"   Оптимизатор: {cfg.train_params.optimizer}, Планировщик: {cfg.train_params.scheduler or 'Нет'}")
        except Exception as e: log.error(f"🛑 Ошибка инициализации criterion/optimizer/scheduler: {e}. Прерывание.", exc_info=True); return None

        results = None
        try:
            results = run_training(
                cfg=cfg, model=model, train_loader=train_loader, val_loader=val_loader,
                criterion=criterion, optimizer=optimizer, scheduler=scheduler, device=device,
                idx2char=idx2char, blank_idx=blank_idx, output_dir=output_dir, trial=None
            )
        except KeyboardInterrupt: log.warning("\n--- ⏹️ Обучение прервано пользователем ---")
        except Exception as e:
            log.error(f"\n--- 🛑 Ошибка во время обучения: {e} ---", exc_info=True)
            try:
                with open(output_dir / "error_log.txt", "w") as f:
                    f.write(f"{datetime.datetime.now()}\nError during training:\n{e}\n"); f.write(traceback.format_exc())
            except Exception as log_e: log.error(f" > Не удалось записать лог ошибки обучения: {log_e}")
            return None

        if results:
            final_lev = results.get('best_val_levenshtein', 'N/A')
            log.info(f"--- ✅ Одиночный Запуск Завершен. Лучший Levenshtein Вал: {final_lev if isinstance(final_lev, str) else f'{final_lev:.4f}'} ---")
            best_model = results.get('final_model'); result_idx2char = results.get('idx2char')
            if best_model and result_idx2char and len(val_loader) > 0:
                log.info("--- 🔬 Пост-тренировочный Анализ ---")
                try:
                    analyze_errors(best_model, val_loader, result_idx2char, blank_idx, device, top_n=10)
                    visualize_predictions(best_model, val_loader, result_idx2char, blank_idx, device, num_samples=5)
                    log.info("--- ✓ Анализ завершен ---")
                except Exception as e: log.error(f"🛑 Ошибка пост-тренировочного анализа: {e}", exc_info=False)
            elif len(val_loader) == 0: log.warning("⚠️ Валидационный загрузчик пуст, пропуск пост-анализа.")
            elif not best_model: log.warning("⚠️ Лучшая модель не была возвращена, пропуск пост-анализа.")

            if 'history' in results:
                try:
                    history_df = pd.DataFrame(results['history']); history_path = output_dir / "training_history.csv"
                    history_df.to_csv(history_path, index=False); log.info(f" > 💾 История обучения сохранена: {history_path.name}")
                    if not history_df.empty:
                        plt.figure(figsize=(12, 5)); plt.subplot(1, 2, 1)
                        if 'train_loss' in history_df: plt.plot(history_df.index, history_df['train_loss'], label='Loss Обуч', marker='.')
                        if 'val_loss' in history_df: plt.plot(history_df.index, history_df['val_loss'], label='Loss Вал', marker='.')
                        plt.title('Потери на Эпохах'); plt.xlabel('Эпоха'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
                        plt.subplot(1, 2, 2)
                        if 'val_levenshtein' in history_df: plt.plot(history_df.index, history_df['val_levenshtein'], label='Lev Вал', marker='.', color='orange')
                        plt.title('Расстояние Левенштейна (Вал)'); plt.xlabel('Эпоха'); plt.ylabel('Levenshtein'); plt.legend(); plt.grid(True)
                        plt.tight_layout(); plot_path = output_dir / "training_plot.png"
                        plt.savefig(plot_path); log.info(f" > 💾 График обучения сохранен: {plot_path.name}"); plt.close()
                except Exception as e: log.error(f"🛑 Не удалось сохранить историю или график: {e}")
            return results.get('best_val_levenshtein')
        else: log.warning("⚠️ Обучение не вернуло результатов."); return None


# --- Точка входа (без изменений) ---
if __name__ == "__main__":
    script_dir = Path(__file__).parent if "__file__" in locals() else Path.cwd()
    CONFIG_DIR_REL = "conf"
    CONFIG_NAME = "config"
    config_path_abs = script_dir / CONFIG_DIR_REL
    if not config_path_abs.is_dir() or not (config_path_abs / f"{CONFIG_NAME}.yaml").exists():
        log.error(f"🛑 Папка конфигурации '{config_path_abs}' или файл '{CONFIG_NAME}.yaml' не найдены.")
    else:
        try:
            hydra.core.global_hydra.GlobalHydra.instance().clear()
            hydra.initialize(config_path=str(config_path_abs.relative_to(Path.cwd())), version_base=None)
            cfg = hydra.compose(config_name=CONFIG_NAME)
            main_runner(cfg)
        except Exception as e:
            log.error(f"🛑 Ошибка инициализации Hydra или запуска: {e}", exc_info=True)

🚀 Старт Запуска: 2025-05-02_19-49-04
 Режим: HPO (Подбор гиперпараметров)
 Корневая папка вывода: hpo_run_2025-05-02_19-49-04
💻 Устройство: cuda
--- 🔍 Настройка Optuna HPO ---
 > Хранилище (Storage): sqlite:///C:\Users\vasja\OneDrive\Рабочий стол\Morse_dev\MorseAudioDecoder\optuna_databases\all_morse_studies.db
 > Имя исследования (Study): 'morse_optimization_20250502_1949'
 > Исследование 'morse_optimization_20250502_1949' создано.
 > Запуск 20 trials...


  0%|          | 0/20 [00:00<?, ?it/s]


--- 📊 Старт Optuna Trial #0 📊 ---
 > ⚙️ Подбор параметров для Trial #0:
   Предложенные параметры:
{   'model_params.block_kernel_size_0': 11,
    'model_params.block_kernel_size_1': 11,
    'model_params.block_stride_0': 4,
    'model_params.block_stride_1': 3,
    'model_params.classifier_dropout': 0.4405603861545576,
    'model_params.rnn_dropout': 0.20085061082888533,
    'model_params.rnn_hidden_size': 256,
    'model_params.rnn_type': 'LSTM',
    'train_params.batch_size': 8,
    'train_params.max_lr': '3.0E-04',
    'train_params.optimizer': 'AdamW',
    'train_params.weight_decay': '7.8E-04'}
 > 💾 Подготовка данных...
   Используется 6000/30000 сэмплов (доля: 0.20).
   Загрузчики данных созданы (Обуч батчей: 675, Вал батчей: 75).
 > 🚀 Инициализация модели и оптимизатора...
--- Создание ResNet блоков ---
  > Блок 0: groups=2 (In=18, Out=64, K=11, S=4)
    -> Длина посл. после блока 0: 1000
  > Блок 1: groups=1 (In=64, Out=128, K=11, S=3)
    -> Длина посл. после блока 1: 334
--

🚂 Обучение:   0%|          | 0/675 [00:00<?, ?it/s]

🧪 Валидация:   0%|          | 0/75 [00:00<?, ?it/s]

...

📊 Эпоха [2/7] | T: 127.2s | Loss(T/V): 0.3340/0.1855 | Lev(V): 0.3667 | LR: 2.8E-04
  ✅ Улучшение Levenshtein (1.8817 -> 0.3667, Δ=1.5150)! Сохраняем модель -> best_model.pth


🚂 Обучение:   0%|          | 0/675 [00:00<?, ?it/s]


--- ⏹️ HPO прерван пользователем ---
--- 🏁 HPO Завершен ---
 > Исследование: 'morse_optimization_20250502_1949'
 > 🏆 Лучший Trial #4: val_levenshtein = 0.2500
   Атрибуты лучшего trial'а:
{   'best_model_absolute_path': 'hpo_run_2025-05-02_19-49-04\\trial_4\\best_model.pth',
    'best_val_levenshtein': 0.25,
    'final_levenshtein_slope': -0.0050000000000000044,
    'final_slope_window': 3}
   Лучшие параметры:
{   'model_params.block_kernel_size_0': 5,
    'model_params.block_kernel_size_1': 7,
    'model_params.block_stride_0': 4,
    'model_params.block_stride_1': 4,
    'model_params.classifier_dropout': 0.43184655198399136,
    'model_params.rnn_dropout': 0.24970538029858527,
    'model_params.rnn_hidden_size': 192,
    'model_params.rnn_type': 'LSTM',
    'train_params.batch_size': 8,
    'train_params.max_lr': 0.0003,
    'train_params.optimizer': 'AdamW',
    'train_params.weight_decay': 1.518795999089651e-05}
 > 💾 Сводка HPO сохранена: hpo_results_summary.csv


[W 2025-05-02 21:01:57,115] Trial 11 failed with parameters: {'model_params.block_kernel_size_0': 7, 'model_params.block_kernel_size_1': 11, 'model_params.block_stride_0': 3, 'model_params.block_stride_1': 4, 'model_params.rnn_type': 'LSTM', 'model_params.rnn_hidden_size': 384, 'model_params.rnn_dropout': 0.3370271220416948, 'model_params.classifier_dropout': 0.24272830699415682, 'train_params.batch_size': 8, 'train_params.max_lr': 0.0003, 'train_params.optimizer': 'AdamW', 'train_params.weight_decay': 0.00024468734669516514} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\vasja\anaconda3\envs\morse_env\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\vasja\AppData\Local\Temp\ipykernel_9140\2569259505.py", line 261, in <lambda>
    study.optimize(lambda trial: objective(trial, cfg.copy()),
  File "C:\Users\vasja\AppData\Local\Temp\ipykernel_9140\2569259505.py

🛑 Не удалось сохранить графики Optuna: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

